**1. Entorno y alcance**
Resuelve la raíz del repositorio e importa el módulo de Sprint 08; no entrena modelos ni ejecuta notebooks previos.

In [1]:
from pathlib import Path
import sys, json, unittest
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/feature_engineering.py').is_file())
sys.path.insert(0, str(ROOT))
import src.auditoria_calidad as ac
pd.set_option('display.width', 220, 'display.max_columns', 40, 'display.max_colwidth', 90, 'display.max_rows', 120)
d, cols, catalogo = ac.load_inputs()
tr = ac.training(d)
print('Panel:', len(d), '| elegibles:', int(d.utilizable_supervisado.sum()), '| entrenamiento:', len(tr),
      '| prueba:', int(d.particion.eq('prueba').sum()), '| atributos E2:', len(cols))
display(tr.groupby('Moneda').agg(filas=('RPJ', 'size'), empresas=('RPJ', 'nunique'), ejercicio_min=('Ejercicio', 'min'),
                                 ejercicio_max=('Ejercicio', 'max'), objetivo_min=('anio_objetivo', 'min'), objetivo_max=('anio_objetivo', 'max')))

Panel: 4221 | elegibles: 3560 | entrenamiento: 2869 | prueba: 691 | atributos E2: 26


,filas,empresas,ejercicio_min,ejercicio_max,objetivo_min,objetivo_max
Moneda,,,,,,
D lares,353,49,2005,2020,2006,2021
Soles,2516,269,2005,2020,2006,2021


Se preservan 4.221 filas, 3.560 elegibles y la partición 2.869/691. `load_inputs` reconstruye con igualdad exacta los atributos financieros y la etiqueta desde `panel_fcl_preliminar.csv` y comprueba que los externos coinciden con sus series anuales por Ejercicio. Entrenamiento: 2.516 filas en soles (269 empresas) y 353 en dólares (49 empresas), ejercicios 2005–2020.

**2. Especificación E2 recuperada de Sprint 06**
Lee por AST el código literal del notebook 07 y el manifiesto `validacion_baseline.json`; contrasta orden, grupos, objetivo y preparación.

In [2]:
spec = ac.especificacion_e2()
display(pd.DataFrame({'orden': range(1, 27), 'atributo': spec['E2'], 'grupo': [ac.grupo_atributo(v, spec) for v in spec['E2']]}).set_index('orden').T)
for clave in ['objetivo', 'preparacion_principal', 'sensibilidades', 'redundantes_excluidas_por_definicion', 'evidencia']:
    print(f'{clave}:', json.dumps(spec[clave], ensure_ascii=False, indent=1))

orden,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26
atributo,pasivo_activo,margen_neto,rentabilidad_activo_cierre,rotacion_activo_cierre,fco_activo,fco_ingresos,inversion_capex_activo,crecimiento_ActivoTotal,crecimiento_TotalIngreso,cambio_FCO_activo_previo,FCO_lag1,FCO_lag2,FCO_lag3,FCO_std3_previa,cambio_FCL_preliminar_activo_previo,FCL_preliminar_lag1,FCL_preliminar_lag2,FCL_preliminar_lag3,FCL_preliminar_std3_previa,inflacion_lima_dic_dic_pct,tipo_cambio_promedio_soles_usd,tasa_referencia_promedio_pct,crecimiento_pbi_real_pct,rentabilidad_indice_anual_pct,volatilidad_diaria_anualizada_pct,monto_renta_variable_anual_millones_pen
grupo,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,financiero,macroeconomico,macroeconomico,macroeconomico,macroeconomico,bursatil,bursatil,bursatil


objetivo: {
 "nombre": "objetivo_fcl_t1",
 "definicion": "FCL_preliminar(t+1) = FCO(t+1) + CAPEX_reportado(t+1) de la misma RPJ, unido por Ejercicio+1 exacto (nunca la fila siguiente); faltante si CAPEX_reportado(t+1) > 0. FCL preliminar, no FCFF validado.",
 "horizonte": "un ejercicio anual adelante (anio_objetivo = Ejercicio + 1)",
 "elegibilidad": "utilizable_supervisado = objetivo no faltante Y Moneda(t) == Moneda(t+1); sin imputación del objetivo",
 "particion": "entrenamiento: anio_objetivo <= 2021 (Ejercicio 2005–2020); prueba: anio_objetivo 2022–2025 (Ejercicio 2021–2024)"
}
preparacion_principal: "Por moneda y ajustado solo en entrenamiento: ColumnTransformer -> SimpleImputer(strategy=\"median\", keep_empty_features=True) -> StandardScaler() (ddof=0) -> Ridge(alpha=1.0, solver=\"svd\")"
sensibilidades: [
 "rezago_externo_t_1: los 7 externos se reemplazan por su valor del año Ejercicio−1 (2005 queda faltante e imputado)",
 "sensibilidad_robust: RobustScaler(quantile_range=(25,7

E2 = 19 financieros + 4 macroeconómicos + 3 bursátiles, en el orden del notebook 07 (celda 10). El objetivo es FCL preliminar de t+1 exacto, sin imputar, elegible solo con la misma moneda. La configuración principal es mediana + StandardScaler por moneda ajustados en entrenamiento; RobustScaler y externos t−1 son sensibilidades, no la especificación principal.

**3. Ejecución de la auditoría y escritura de reportes**
`run()` recalcula todos los diagnósticos solo con entrenamiento, escribe `reports/sprint_08/` y falla si cambia algún archivo previo durante la ejecución.

In [3]:
tablas = ac.run()
display(pd.Series({k: len(v) for k, v in tablas.items()}, name='filas').to_frame())

,filas
calidad_atributos,52
cobertura_temporal_train,832
faltantes_causas_train,108
denominadores_train,66
denominadores_casos_train,549
saltos_escala_train,23
correlaciones_pearson,52
correlaciones_spearman,52
n_pares_correlacion,52
pares_correlacionados,12


Se generan los ocho entregables y los anexos de trazabilidad. Las comprobaciones previas a la escritura (especificación, reconstrucción, conteos) y la comparación de hashes antes/después de la ejecución se superaron.

**4. Diccionario de atributos**
Nombre, definición, unidad, escala, año de referencia, fuente original y evidencia con archivo y línea o celda.

In [4]:
display(tablas['diccionario_atributos'][['orden', 'nombre', 'grupo', 'unidad', 'escala', 'anio_referencia', 'fuente_original', 'evidencia']])

,orden,nombre,grupo,unidad,escala,anio_referencia,fuente_original,evidencia
0,1,pasivo_activo,financiero,adimensional,fracción decimal (0.05 = 5 %),t,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:16; reports/diccionario_integracion_bursatil.csv:27; notebo...
1,2,margen_neto,financiero,adimensional,fracción decimal (0.05 = 5 %),t,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:17; reports/diccionario_integracion_bursatil.csv:28; notebo...
2,3,rentabilidad_activo_cierre,financiero,adimensional,fracción decimal (0.05 = 5 %),t,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:18; reports/diccionario_integracion_bursatil.csv:29; notebo...
3,4,rotacion_activo_cierre,financiero,adimensional,fracción decimal (0.05 = 5 %),t,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:19; reports/diccionario_integracion_bursatil.csv:30; notebo...
4,5,fco_activo,financiero,adimensional,fracción decimal (0.05 = 5 %),t,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:20; reports/diccionario_integracion_bursatil.csv:31; notebo...
5,6,fco_ingresos,financiero,adimensional,fracción decimal (0.05 = 5 %),t,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:21; reports/diccionario_integracion_bursatil.csv:32; notebo...
6,7,inversion_capex_activo,financiero,adimensional,fracción decimal (0.05 = 5 %),t,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:22; reports/diccionario_integracion_bursatil.csv:33; notebo...
7,8,crecimiento_ActivoTotal,financiero,adimensional,fracción decimal (0.10 = 10 %),t y t-1,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:82; reports/diccionario_integracion_bursatil.csv:35; notebo...
8,9,crecimiento_TotalIngreso,financiero,adimensional,fracción decimal (0.10 = 10 %),t y t-1,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:82; reports/diccionario_integracion_bursatil.csv:36; notebo...
9,10,cambio_FCO_activo_previo,financiero,adimensional,fracción decimal del activo t-1,t y t-1,SMV: estados financieros individuales anuales (data/processed/panel_fcl_preliminar.csv...,src/feature_engineering.py:86; reports/diccionario_integracion_bursatil.csv:37; notebo...


Los ratios, crecimientos y cambios son fracciones decimales; los macroeconómicos en porcentaje están en puntos porcentuales; los rezagos y volatilidades están en unidades monetarias reportadas, cuya escala (unidades o miles) no está acreditada. Cada fila remite a la línea de `src/feature_engineering.py` o de los diccionarios BCRP que la define.

**5. Calidad por atributo y moneda (entrenamiento)**
Descriptivos sin imputar, percentiles lineales, extremos por RIQ, constantes, infinitos e indefinidos; faltantes desglosados por causa.

In [5]:
q = tablas['calidad_atributos']
display(q[['moneda', 'variable', 'n', 'faltantes_pct', 'faltantes_estructurales', 'faltantes_regla', 'faltantes_fuente', 'minimo', 'p01', 'p50', 'p99',
           'maximo', 'asimetria', 'extremos_RIQ', 'es_constante', 'infinitos', 'unidad_estadistica']].round(3))
print('Constantes:', int(q.es_constante.sum()), '| infinitos:', int(q.infinitos.sum()), '| faltantes sin causa:', int(q.faltantes_no_explicados.sum()))

,moneda,variable,n,faltantes_pct,faltantes_estructurales,faltantes_regla,faltantes_fuente,minimo,p01,p50,p99,maximo,asimetria,extremos_RIQ,es_constante,infinitos,unidad_estadistica
0,D lares,pasivo_activo,353,0.000,0,0,0,0.000,0.001,0.469,0.952,1.370,0.032,1,False,0,empresa-ejercicio
1,D lares,margen_neto,353,7.649,0,27,0,-1.353,-0.532,0.133,0.998,1.232,0.167,22,False,0,empresa-ejercicio
2,D lares,rentabilidad_activo_cierre,353,0.000,0,0,0,-0.978,-0.174,0.049,0.563,0.697,0.192,43,False,0,empresa-ejercicio
3,D lares,rotacion_activo_cierre,353,7.649,0,27,0,0.010,0.028,0.419,3.576,4.913,2.514,28,False,0,empresa-ejercicio
4,D lares,fco_activo,353,2.833,0,0,10,-2.528,-0.173,0.093,0.623,0.942,-5.643,29,False,0,empresa-ejercicio
5,D lares,fco_ingresos,353,10.198,0,26,10,-0.692,-0.230,0.284,1.544,2.005,1.769,14,False,0,empresa-ejercicio
6,D lares,inversion_capex_activo,353,0.000,0,0,0,-0.000,-0.000,0.028,0.310,0.365,2.609,16,False,0,empresa-ejercicio
7,D lares,crecimiento_ActivoTotal,353,14.731,52,0,0,-0.897,-0.361,0.032,0.758,2.286,3.412,19,False,0,empresa-ejercicio
8,D lares,crecimiento_TotalIngreso,353,19.830,52,18,0,-0.717,-0.609,0.030,3.601,10.268,8.961,19,False,0,empresa-ejercicio
9,D lares,cambio_FCO_activo_previo,353,20.113,52,0,19,-0.305,-0.288,0.002,0.365,2.529,8.379,34,False,0,empresa-ejercicio


Constantes: 0 | infinitos: 0 | faltantes sin causa: 0


No hay constantes ni infinitos. En soles, varios ratios y crecimientos tienen asimetrías de 20 a 50 por colas extremas; en dólares, las colas son moderadas. Los extremos RIQ de los externos (por ejemplo, 149 en inflación) son artefactos de repetir 16 valores anuales entre empresas, no observaciones atípicas independientes.

**6. Cobertura temporal y causas de faltantes**
Cada faltante se clasifica según las reglas de Sprint 06: estructural (historia contigua insuficiente), regla de cálculo o fuente ausente.

In [6]:
causas = tablas['faltantes_causas_train']
display(causas.pivot_table(index=['moneda', 'variable'], columns='grupo_causa', values='n', aggfunc='sum', fill_value=0))
display(causas.groupby(['grupo_causa', 'causa']).n.sum().to_frame())
cov = tablas['cobertura_temporal_train']
display(cov.query("variable == 'FCO_lag3' and moneda == 'Soles'")[['anio', 'n', 'faltantes', 'faltantes_estructurales', 'faltantes_fuente']].set_index('anio').T)

grupo_causa                                  estructural  fuente  regla
moneda  variable                                                       
D lares FCL_preliminar_lag1                           52      16      0
        FCL_preliminar_lag2                           97      15      0
        FCL_preliminar_lag3                          139      15      0
        FCL_preliminar_std3_previa                   139      30      0
        FCO_lag1                                      52      16      0
        FCO_lag2                                      97      15      0
        FCO_lag3                                     139      15      0
        FCO_std3_previa                              139      30      0
        cambio_FCL_preliminar_activo_previo           52      19      0
        cambio_FCO_activo_previo                      52      19      0
        crecimiento_ActivoTotal                       52       0      0
        crecimiento_TotalIngreso                      52       0     18
        fco_activo                                     0      10      0
        fco_ingresos                                   0      10     26
        margen_neto                                    0       0     27
        rotacion_activo_cierre                         0       0     27
Soles   FCL_preliminar_lag1                          243     127      0
        FCL_preliminar_lag2                          466     153      0
        FCL_preliminar_lag3                          679     176      0
        FCL_preliminar_std3_previa                   679     233      0
        FCO_lag1                                     243      46      0
        FCO_lag2                                     466      43      0
        FCO_lag3                                     679      41      0
        FCO_std3_previa                              679      82      0
        cambio_FCL_preliminar_activo_previo          243     149      0
        cambio_FCO_activo_previo                     243      60      0
        crecimiento_ActivoTotal                      243       0      0
        crecimiento_TotalIngreso                     243       0    133
        fco_activo                                     0      41      0
        fco_ingresos                                   0      41    140
        inversion_capex_activo                         0      50     12
        margen_neto                                    0       0    141
        rotacion_activo_cierre                         0       0    141

n
grupo_causa causa                                                                  
estructural estructural: entrada de la empresa al panel después de 2005        2046
            estructural: historia anterior a 2005 fuera del panel              3140
            estructural: ruptura de segmento (hueco anual o cambio de moneda)   982
fuente      fuente: flujo ausente en la ventana t-3..t-1                        375
            fuente: flujo ausente en t o t-1                                    247
            fuente: flujo ausente en t-1                                        205
            fuente: flujo ausente en t-2                                        226
            fuente: flujo ausente en t-3                                        247
            fuente: numerador o denominador ausente en t                        152
regla       regla: CAPEX positivo excluido                                       12
            regla: base t-1 no positiva                                         135
            regla: denominador no positivo en t                                 334
            regla: ingreso actual no positivo                                    16
            regla: ingreso no positivo en t                                     168

anio,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
n,139,138,143,149,137,126,143,180,178,179,175,177,169,162,161,160
faltantes,139,138,143,38,22,19,19,26,36,41,19,25,21,16,9,9
faltantes_estructurales,139,138,143,27,22,19,19,26,24,23,19,25,21,16,9,9
faltantes_fuente,0,0,0,11,0,0,0,0,12,18,0,0,0,0,0,0


Las causas reconstruidas cubren las 8.285 celdas faltantes de entrenamiento. En rezagos de 3 años y ventanas, la mayoría es estructural: el panel empieza en 2005, por lo que en 2005–2007 faltan por construcción. Ratios y crecimientos faltan sobre todo por las reglas de denominador o ingreso no positivo, no por ausencia en la fuente.

**7. Denominadores positivos pequeños**
Criterios exploratorios transversal (fracción de la mediana de la moneda) e intraempresa (fracción de la mediana de la propia empresa), con sensibilidad 0,1 %, 1 % y 5 %.

In [7]:
den = tablas['denominadores_train']
display(den.pivot_table(index=['moneda', 'variable', 'denominador'], columns='fraccion', values=['casos_transversal', 'casos_intraempresa'], fill_value=0))
casos = tablas['denominadores_casos_train']
display(casos.loc[casos.criterio_intraempresa | casos.relativo_moneda.lt(1e-3)].sort_values('relativo_moneda')
        [['moneda', 'variable', 'RPJ', 'NombreEmpresa', 'Ejercicio', 'denominador', 'valor_denominador', 'relativo_moneda', 'relativo_empresa', 'valor_atributo', 'atributo_extremo_RIQ']].head(25))

casos_intraempresa             casos_transversal             
fraccion                                                                   0.001 0.010 0.050             0.001 0.010  0.050
moneda  variable                            denominador                                                                    
D lares cambio_FCL_preliminar_activo_previo ActivoTotal(t-1)                 0.0   0.0   0.0               0.0   0.0    6.0
        cambio_FCO_activo_previo            ActivoTotal(t-1)                 0.0   0.0   0.0               0.0   0.0    6.0
        crecimiento_ActivoTotal             ActivoTotal(t-1)                 0.0   0.0   0.0               0.0   0.0    6.0
        crecimiento_TotalIngreso            TotalIngreso(t-1)                0.0   0.0   0.0               0.0   2.0    8.0
        fco_activo                          ActivoTotal(t)                   0.0   0.0   0.0               0.0   0.0    8.0
        fco_ingresos                        TotalIngreso(t)                  0.0   0.0   0.0               0.0   2.0    8.0
        inversion_capex_activo              ActivoTotal(t)                   0.0   0.0   0.0               0.0   0.0    8.0
        margen_neto                         TotalIngreso(t)                  0.0   0.0   0.0               0.0   2.0    8.0
        pasivo_activo                       ActivoTotal(t)                   0.0   0.0   0.0               0.0   0.0    8.0
        rentabilidad_activo_cierre          ActivoTotal(t)                   0.0   0.0   0.0               0.0   0.0    8.0
        rotacion_activo_cierre              ActivoTotal(t)                   0.0   0.0   0.0               0.0   0.0    8.0
Soles   cambio_FCL_preliminar_activo_previo ActivoTotal(t-1)                 1.0   1.0   4.0               3.0  38.0  125.0
        cambio_FCO_activo_previo            ActivoTotal(t-1)                 1.0   1.0   4.0               3.0  38.0  125.0
        crecimiento_ActivoTotal             ActivoTotal(t-1)                 1.0   1.0   4.0               3.0  38.0  125.0
        crecimiento_TotalIngreso            TotalIngreso(t-1)                1.0   7.0  29.0              24.0  60.0  212.0
        fco_activo                          ActivoTotal(t)                   1.0   1.0   7.0               4.0  45.0  146.0
        fco_ingresos                        TotalIngreso(t)                  1.0   8.0  34.0              24.0  71.0  244.0
        inversion_capex_activo              ActivoTotal(t)                   1.0   1.0   7.0               4.0  45.0  146.0
        margen_neto                         TotalIngreso(t)                  1.0   8.0  34.0              24.0  71.0  244.0
        pasivo_activo                       ActivoTotal(t)                   1.0   1.0   7.0               4.0  45.0  146.0
        rentabilidad_activo_cierre          ActivoTotal(t)                   1.0   1.0   7.0               4.0  45.0  146.0
        rotacion_activo_cierre              ActivoTotal(t)                   1.0   1.0   7.0               4.0  45.0  146.0

,moneda,variable,RPJ,NombreEmpresa,Ejercicio,denominador,valor_denominador,relativo_moneda,relativo_empresa,valor_atributo,atributo_extremo_RIQ
44,Soles,pasivo_activo,OE6928,FOSSAL S.A.A.,2016,ActivoTotal(t),1.0,0.000002,0.000004,17.000000,True
253,Soles,fco_activo,OE6928,FOSSAL S.A.A.,2016,ActivoTotal(t),1.0,0.000002,0.000004,0.000000,False
208,Soles,rotacion_activo_cierre,OE6928,FOSSAL S.A.A.,2016,ActivoTotal(t),1.0,0.000002,0.000004,NaN,False
163,Soles,rentabilidad_activo_cierre,OE6928,FOSSAL S.A.A.,2016,ActivoTotal(t),1.0,0.000002,0.000004,-17.000000,True
372,Soles,inversion_capex_activo,OE6928,FOSSAL S.A.A.,2016,ActivoTotal(t),1.0,0.000002,0.000004,-0.000000,False
548,Soles,cambio_FCL_preliminar_activo_previo,OE6928,FOSSAL S.A.A.,2017,ActivoTotal(t-1),1.0,0.000002,0.000004,385.000000,True
410,Soles,crecimiento_ActivoTotal,OE6928,FOSSAL S.A.A.,2017,ActivoTotal(t-1),1.0,0.000002,0.000004,269793.000000,True
510,Soles,cambio_FCO_activo_previo,OE6928,FOSSAL S.A.A.,2017,ActivoTotal(t-1),1.0,0.000002,0.000004,385.000000,True
436,Soles,crecimiento_TotalIngreso,B80049,GR HOLDING S.A.,2016,TotalIngreso(t-1),3.0,0.000017,0.000760,25.666667,True
282,Soles,fco_ingresos,B80049,GR HOLDING S.A.,2015,TotalIngreso(t),3.0,0.000017,0.000760,5504.666667,True


El criterio transversal identifica sobre todo empresas pequeñas, cuyos ratios suelen ser normales. El intraempresa aísla caídas anómalas: FOSSAL 2016 tiene ActivoTotal = 1, con el que su crecimiento de 2017 vale 269.793, y GR Holding 2015 tiene TotalIngreso = 3, con el que su margen neto vale 4.958. Son casos por conciliar con la fuente, no errores declarados.

**8. Saltos de magnitud intraempresa**
Cambios ≥ 100× entre años consecutivos en montos positivos, restringidos a ejercicios ≤ 2020; "pico" si el cambio revierte al año siguiente.

In [8]:
display(tablas['saltos_escala_train'])

,moneda,RPJ,NombreEmpresa,Ejercicio,columna,valor_t_1,valor_t,valor_t_mas_1,factor,patron_pico,fila_t_en_entrenamiento,ejercicios_train_afectados,revision_interanual
0,Soles,CD0009,TRANSACCIONES FINANCIERAS S.A.,2016,ActivoTotal,60862.0,67619923.0,74347.0,1111.036821,True,True,2016;2017,True
1,Soles,CD0009,TRANSACCIONES FINANCIERAS S.A.,2017,ActivoTotal,67619923.0,74347.0,90657.0,0.001099,False,True,2017,True
2,Soles,OE3108,EDYFICAR PERU S.A.,2015,ActivoTotal,5210448.0,21430.0,11789.0,0.004113,False,True,2015,False
3,Soles,OE6928,FOSSAL S.A.A.,2017,ActivoTotal,1.0,269794.0,265748.0,269794.000000,False,True,2017;2018;2019;2020,False
4,Soles,020045,ACTIVO INMOBILIARIO PERUANO S.A.A. (ANTES FALABELLA PERU S.A.A.),2015,PasivoTotal,619.0,532983.0,1324098.0,861.038772,False,True,2015;2016;2017;2018,True
5,Soles,B80002,GESTIONES Y RECUPERACIONES DE ACTIVOS SOCIEDAD ANONIMA (RET),2006,PasivoTotal,278357.0,551.0,155.0,0.001979,False,True,2006;2007,False
6,Soles,B80031,LATINA HOLDING S.A.,2016,PasivoTotal,18.0,14938.0,6480.0,829.888889,False,True,2016;2017,False
7,Soles,CD0001,CFI HOLDINGS S.A.,2006,PasivoTotal,117.0,26589.0,1025.0,227.256410,False,True,2006;2007;2008;2009,False
8,Soles,CD0009,TRANSACCIONES FINANCIERAS S.A.,2016,PasivoTotal,217.0,320687.0,129.0,1477.820276,True,True,2016;2017,True
9,Soles,CD0009,TRANSACCIONES FINANCIERAS S.A.,2017,PasivoTotal,320687.0,129.0,176.0,0.000402,False,True,2017,True


Hay 23 saltos que afectan filas de entrenamiento. Solo CD0009 2016 muestra un pico (×1.111 en activo, revertido en 2017), compatible con un cambio de unidad aunque no probado. Otros saltos coinciden con eventos plausibles (por ejemplo, Edyficar 2015), lo que confirma que no conviene tratarlos como errores sin documentación.

**9. Correlaciones de Pearson y Spearman y tamaños efectivos**
Pares con |r| ≥ 0,8 con n, empresas y años; Pearson leave-one-out para detectar dependencia de un caso; los externos se correlacionan además sobre sus 16 años.

In [9]:
display(tablas['pares_correlacionados'][['moneda', 'variable_1', 'variable_2', 'pearson', 'spearman', 'n_pares', 'n_empresas', 'n_anios',
                                          'pearson_sin_caso_influyente', 'caso_influyente', 'depende_de_un_caso']].round(4))
an = tablas['correlaciones_externas_anuales']
display(an.loc[an.moneda.eq('Soles')].sort_values('pearson_anual', key=abs, ascending=False).head(6).round(3))

,moneda,variable_1,variable_2,pearson,spearman,n_pares,n_empresas,n_anios,pearson_sin_caso_influyente,caso_influyente,depende_de_un_caso
0,D lares,margen_neto,rentabilidad_activo_cierre,0.6461,0.8230,326,41,16,0.6018,CM0007 2020 (NEXA RESOURCES ATACOCHA S.A.A.),False
1,D lares,cambio_FCO_activo_previo,cambio_FCL_preliminar_activo_previo,0.9669,0.9074,282,43,15,0.9212,OE4860 2020 (ENERGIA DEL PACIFICO S.A.),False
2,D lares,FCO_std3_previa,FCL_preliminar_std3_previa,0.9263,0.9392,184,32,13,0.8949,CM0006 2018 (SOCIEDAD MINERA CERRO VERDE S.A.A.),False
3,Soles,margen_neto,fco_ingresos,0.8268,0.4513,2335,254,16,0.2845,B80049 2015 (GR HOLDING S.A.),True
4,Soles,fco_activo,inversion_capex_activo,-0.9212,0.2789,2427,267,16,0.1147,OE3108 2015 (EDYFICAR PERU S.A.),True
5,Soles,crecimiento_ActivoTotal,cambio_FCO_activo_previo,0.9996,0.0709,2213,251,15,-0.6713,OE6928 2017 (FOSSAL S.A.A.),True
6,Soles,crecimiento_ActivoTotal,cambio_FCL_preliminar_activo_previo,0.9995,0.0115,2124,250,15,-0.6365,OE6928 2017 (FOSSAL S.A.A.),True
7,Soles,cambio_FCO_activo_previo,cambio_FCL_preliminar_activo_previo,0.9999,0.9256,2124,250,15,0.9125,OE6928 2017 (FOSSAL S.A.A.),False
8,Soles,FCO_lag1,FCL_preliminar_lag1,0.9602,0.8683,2146,250,15,0.9510,B80005 2018 (BANCO DE CREDITO DEL PERU),False
9,Soles,FCO_lag2,FCL_preliminar_lag2,0.9591,0.8680,1897,238,14,0.9460,B80005 2019 (BANCO DE CREDITO DEL PERU),False


,moneda,variable_1,variable_2,pearson_anual,spearman_anual,n_anios,pearson_filas,spearman_filas
32,Soles,tasa_referencia_promedio_pct,crecimiento_pbi_real_pct,0.716,0.576,16,0.742,0.602
24,Soles,inflacion_lima_dic_dic_pct,rentabilidad_indice_anual_pct,-0.679,-0.674,16,-0.666,-0.651
22,Soles,inflacion_lima_dic_dic_pct,tasa_referencia_promedio_pct,0.554,0.441,16,0.547,0.442
25,Soles,inflacion_lima_dic_dic_pct,volatilidad_diaria_anualizada_pct,0.534,0.291,16,0.530,0.306
28,Soles,tipo_cambio_promedio_soles_usd,crecimiento_pbi_real_pct,-0.487,-0.403,16,-0.491,-0.410
27,Soles,tipo_cambio_promedio_soles_usd,tasa_referencia_promedio_pct,-0.415,-0.374,16,-0.443,-0.400


Cuatro de los 9 pares altos en soles (12 en total) dependen de un solo caso: la correlación de Pearson de 0,9996 entre crecimiento del activo y cambio de FCO pasa a −0,67 sin FOSSAL 2017, y su Spearman es 0,07. Son estables los pares entre rezagos FCO/FCL del mismo año (Pearson ~0,96, Spearman ~0,87) y entre volatilidades. Entre externos, ningún par anual supera 0,8 y el tamaño efectivo es de 16 años.

**10. VIF, condicionamiento, constantes, duplicados y dependencias**
VIF en la matriz E2 imputada y estandarizada, en casos completos, sin casos influyentes y en el bloque externo anual; SVD con tolerancia explícita.

In [10]:
vif = tablas['diagnostico_vif']
display(vif.pivot_table(index='variable', columns=['moneda', 'especificacion'], values='VIF').round(1))
display(vif.groupby(['moneda', 'especificacion'])[['n', 'n_anios', 'rango', 'numero_condicion']].first())
display(tablas['dependencias_lineales'])

moneda                                          D lares                                              Soles                                                                  
especificacion                          casos_completos externos_anual imputada_mediana_E2 casos_completos externos_anual imputada_mediana_E2 imputada_sin_casos_influyentes
variable                                                                                                                                                                    
FCL_preliminar_lag1                                 6.5            NaN                 4.7            47.7            NaN                23.4                           23.4
FCL_preliminar_lag2                                10.3            NaN                 7.4            84.7            NaN                29.0                           28.9
FCL_preliminar_lag3                                16.7            NaN                12.6            66.2            NaN                23.1                           23.1
FCL_preliminar_std3_previa                         13.5            NaN                12.4           111.0            NaN                10.6                           10.6
FCO_lag1                                           14.0            NaN                 6.6            55.1            NaN                28.1                           28.1
FCO_lag2                                           15.0            NaN                 9.6            98.5            NaN                35.8                           35.7
FCO_lag3                                           20.4            NaN                13.1            76.8            NaN                28.8                           28.8
FCO_std3_previa                                    10.8            NaN                 9.4           111.3            NaN                11.2                           11.2
cambio_FCL_preliminar_activo_previo                14.0            NaN                24.6            15.2            NaN              7519.2                            5.8
cambio_FCO_activo_previo                           12.9            NaN                23.4            14.8            NaN              7942.6                            5.9
crecimiento_ActivoTotal                             1.4            NaN                 1.3             1.2            NaN              1268.7                            2.1
crecimiento_TotalIngreso                            1.4            NaN                 1.1             1.0            NaN                 1.0                            1.0
crecimiento_pbi_real_pct                            4.2            3.0                 4.0             4.4            3.0                 3.2                            3.2
fco_activo                                          2.9            NaN                 1.8             2.7            NaN                10.6                            1.4
fco_ingresos                                        2.3            NaN                 1.6             6.4            NaN                 6.9                            1.3
inflacion_lima_dic_dic_pct                          6.7            5.8                 6.4             7.9            5.8                 5.6                            5.6
inversion_capex_activo                              2.3            NaN                 1.9             2.2            NaN                 9.8                            1.5
margen_neto                                         3.0            NaN                 2.3             6.5            NaN                 5.7                            1.3
monto_renta_variable_anual_millones_pen             1.8            1.3                 1.4             1.5            1.3                 1.3                            1.3
pasivo_activo                                       1.5            NaN                 1.3             1.2            NaN                 1.1                            1.1
rentabilidad_activo_c

n  n_anios  rango  numero_condicion
moneda  especificacion                                                        
D lares casos_completos                  176       13     26         13.749420
        externos_anual                    16       16      7          5.675469
        imputada_mediana_E2              353       16     26         14.008407
Soles   casos_completos                 1498       13     26         32.373747
        externos_anual                    16       16      7          5.675469
        imputada_mediana_E2             2516       16     26        221.433703
        imputada_sin_casos_influyentes  2513       16     26         18.389222

,moneda,especificacion,tipo,indice_condicion,variables,detalle,tolerancia
0,Soles,imputada_mediana_E2,"casi dependencia (índice de condición, exploratorio)",221.433703,cambio_FCO_activo_previo;cambio_FCL_preliminar_activo_previo,"{""cambio_FCO_activo_previo"": 0.9641, ""cambio_FCL_preliminar_activo_previo"": 0.9539}",30
1,Soles,casos_completos,"casi dependencia (índice de condición, exploratorio)",32.373747,FCO_lag2;FCO_lag3;FCL_preliminar_lag2;FCL_preliminar_lag3,"{""FCO_lag2"": 0.9467, ""FCO_lag3"": 0.6384, ""FCL_preliminar_lag2"": 0.9342, ""FCL_prelimina...",30


El número de condición imputado (soles 221,43; dólares 14,01) coincide con Sprint 06. En soles, al excluir tres casos baja a 18,39, y el VIF de los cambios FCO/FCL pasa de ~7.900 a ~6: su redundancia aparente depende de esos casos. Los rezagos FCO/FCL mantienen VIF de 23 a 36 imputados y de 48 a 111 en casos completos (la mediana atenúa la colinealidad). No hay constantes, duplicados ni dependencias exactas: el rango es 26 en ambas monedas.

**11. Auditoría temporal por atributo**
Distingue año de referencia, fecha de publicación, vintage y fecha de pronóstico; clasifica cada atributo sin certificar disponibilidad por la sola alineación t → t+1.

In [11]:
tem = tablas['auditoria_temporal']
display(tem[['variable', 'anio_referencia', 'clasificacion', 'pct_filas_train_con_revision_interanual', 'clasificacion_si_pronostico_31dic_t']].round(1))
display(tem.clasificacion.value_counts().to_frame())

,variable,anio_referencia,clasificacion,pct_filas_train_con_revision_interanual,clasificacion_si_pronostico_31dic_t
0,pasivo_activo,t,POSIBLE FUGA TEMPORAL,35.1,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
1,margen_neto,t,POSIBLE FUGA TEMPORAL,35.9,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
2,rentabilidad_activo_cierre,t,POSIBLE FUGA TEMPORAL,35.1,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
3,rotacion_activo_cierre,t,POSIBLE FUGA TEMPORAL,35.9,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
4,fco_activo,t,POSIBLE FUGA TEMPORAL,35.2,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
5,fco_ingresos,t,POSIBLE FUGA TEMPORAL,36.1,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
6,inversion_capex_activo,t,POSIBLE FUGA TEMPORAL,34.5,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
7,crecimiento_ActivoTotal,t y t-1,POSIBLE FUGA TEMPORAL,34.5,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
8,crecimiento_TotalIngreso,t y t-1,POSIBLE FUGA TEMPORAL,35.5,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)
9,cambio_FCO_activo_previo,t y t-1,POSIBLE FUGA TEMPORAL,34.5,POSIBLE FUGA TEMPORAL (EEFF de t no publicados al 31/12/t)


,count
clasificacion,
DISPONIBILIDAD NO VERIFICADA,14
POSIBLE FUGA TEMPORAL,12


Ningún atributo queda verificado: no hay fechas de publicación por fila, vintages ni fecha de pronóstico definida. Los 11 financieros que usan montos de t y el crecimiento del PBI son posible fuga (reexpresiones en el 33–36 % de las filas y revisiones del PBI). Los rezagos, las ventanas y los demás externos quedan como no verificados.

**12. Hallazgos y tabla final de los 26 atributos**
Separa problemas comprobados de sospechas pendientes; la recomendación es provisional y no usa resultados de prueba.

In [12]:
display(tablas['hallazgos_calidad'][['id', 'estado', 'categoria', 'moneda', 'evidencia']])
display(tablas['tabla_26_atributos'][['nombre', 'origen', 'riesgo_temporal', 'redundancia', 'recomendacion']])

,id,estado,categoria,moneda,evidencia
0,H01,COMPROBADO,correlación dominada por un caso,Soles,Pearson=0.826788 (n=2335); sin B80049 2015 (GR HOLDING S.A.): 0.2845; Spearman=0.4513
1,H02,COMPROBADO,correlación dominada por un caso,Soles,Pearson=-0.921203 (n=2427); sin OE3108 2015 (EDYFICAR PERU S.A.): 0.1147; Spearman=0.2789
2,H03,COMPROBADO,correlación dominada por un caso,Soles,Pearson=0.999586 (n=2213); sin OE6928 2017 (FOSSAL S.A.A.): -0.6713; Spearman=0.0709
3,H04,COMPROBADO,correlación dominada por un caso,Soles,Pearson=0.999544 (n=2124); sin OE6928 2017 (FOSSAL S.A.A.): -0.6365; Spearman=0.0115
4,H05,COMPROBADO,denominador positivo pequeño,D lares,"6 celdas (3 empresa-ejercicio) con fracción 0.01: transversal 6, intraempresa 0, con a..."
5,H06,COMPROBADO,denominador positivo pequeño,Soles,"543 celdas (142 empresa-ejercicio) con fracción 0.01: transversal 541, intraempresa 31..."
6,H07,COMPROBADO,salto intraempresa de magnitud,D lares;Soles,23 cambios >= 100x entre años consecutivos (2 con reversión al año siguiente): OE6928 ...
7,H08,COMPROBADO,faltantes estructurales por rezago,ambas,D lares FCO_lag3: 139/154; D lares FCO_std3_previa: 139/169; D lares FCL_preliminar_la...
8,H09,COMPROBADO,indefinidos por regla de cálculo,ambas,celdas: D lares regla: base t-1 no positiva: 16; D lares regla: denominador no positiv...
9,H10,COMPROBADO,control de causas de faltantes,ambas,8285 celdas faltantes en train; no explicadas por las reglas de Sprint 06: 0


,nombre,origen,riesgo_temporal,redundancia,recomendacion
0,pasivo_activo,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: ninguno | VIF imputado/casos completos: D lares 1.3/1.5; Soles 1.1/1.2,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
1,margen_neto,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: Soles fco_ingresos P=0.83/S=0.45 (un caso); D lares rentabilidad_activ...,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
2,rentabilidad_activo_cierre,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: D lares margen_neto P=0.65/S=0.82 | VIF imputado/casos completos: D la...,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
3,rotacion_activo_cierre,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: ninguno | VIF imputado/casos completos: D lares 1.4/1.5; Soles 1.2/1.5,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
4,fco_activo,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: Soles inversion_capex_activo P=-0.92/S=0.28 (un caso) | VIF imputado/c...,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
5,fco_ingresos,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: Soles margen_neto P=0.83/S=0.45 (un caso) | VIF imputado/casos complet...,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
6,inversion_capex_activo,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: Soles fco_activo P=-0.92/S=0.28 (un caso) | VIF imputado/casos complet...,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
7,crecimiento_ActivoTotal,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: Soles cambio_FCO_activo_previo P=1.00/S=0.07 (un caso); Soles cambio_F...,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
8,crecimiento_TotalIngreso,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: ninguno | VIF imputado/casos completos: D lares 1.1/1.4; Soles 1.0/1.0,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...
9,cambio_FCO_activo_previo,financiero: SMV: estados financieros individuales anuales,POSIBLE FUGA TEMPORAL,pares |r|>=0.8: Soles cambio_FCL_preliminar_activo_previo P=1.00/S=0.93; Soles crecimi...,Conservar provisionalmente; verificar reexpresiones (revision_interanual) antes del mo...


Hay 16 hallazgos comprobados (casos influyentes, denominadores, saltos, causas de faltantes, VIF y condicionamiento, y controles sin hallazgo) y 5 sospechas pendientes (escala de CD0009 2016, reexpresiones, revisiones del PBI, escala monetaria y quiebre bursátil de 2015). Los 26 atributos se conservan provisionalmente, cada uno con sus condiciones de verificación.

**13. Integridad de archivos previos**
Compara los archivos rastreados con HEAD y todos los archivos ajenos a Sprint 08 con la referencia tomada antes de editar.

In [13]:
integridad = ac.verificar_integridad()
print({k: (len(v) if isinstance(v, list) else v) for k, v in integridad.items() if k not in ('alcance', 'referencia_descripcion')})
print(integridad['alcance'])

{'referencia': 'reports/sprint_08/referencia_integridad_previa.csv', 'archivos_referencia': 1505, 'modificados': 0, 'ausentes': 0, 'nuevos_fuera_de_sprint08': 0, 'git_head': 'b2868852000a388cb9d6fd6abafc52c14a27691d', 'git_rastreados_modificados': 0}
Para archivos rastreados, git status contra HEAD acredita que no cambiaron desde el último commit. Para no rastreados (Sprint 07 y evidencia sectorial), la referencia solo acredita ausencia de cambios desde el inicio de esta revisión.


Ningún archivo rastreado difiere de HEAD y ninguno de los 1.505 archivos de la referencia cambió ni falta. Para los archivos no rastreados de Sprint 07, la referencia solo acredita que no cambiaron desde el inicio de esta revisión.

**14. Pruebas automatizadas de Sprint 08**
Ejecuta las pruebas de especificación, aislamiento con controles positivos, VIF, causas, temporalidad, reproducibilidad e integridad (la estructura del notebook se prueba fuera de él).

In [14]:
suite = unittest.defaultTestLoader.loadTestsFromName('tests.test_auditoria_calidad')
def aplanar(s):
    for t in s:
        yield from (aplanar(t) if isinstance(t, unittest.TestSuite) else [t])
seleccion = unittest.TestSuite(t for t in aplanar(suite) if 'notebook' not in t.id())
resultado = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(seleccion)
assert resultado.wasSuccessful()

.

.

.

.

.

.

... (6 outputs left out)

.

.

.

----------------------------------------------------------------------
Ran 15 tests in 18.522s

OK


Las pruebas confirman que perturbar las filas de prueba y los montos futuros no cambia ningún diagnóstico, y que los dos controles contaminados sí los cambian; también confirman la reproducción byte a byte y la integridad de los archivos previos.